In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cpu"

# Dataset sizes
TRAIN_SAMPLES_PER_KEY = 2000
VAL_SAMPLES_PER_KEY = 400

# Sequence config
SEQ_LEN = 64          # timesteps per example
CHUNK_MIN = 2         # min timesteps per event (chord or note)
CHUNK_MAX = 6         # max timesteps per event

# Model config
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

PITCH_CLASSES = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
KEY_TO_INDEX = {f"{p}_major": i for i, p in enumerate(PITCH_CLASSES)}
KEY_TO_INDEX.update({f"{p}_minor": i+12 for i, p in enumerate(PITCH_CLASSES)})

def key_index_to_name(idx: int) -> str:
    if idx < 12:
        return f"{PITCH_CLASSES[idx]}_major"
    return f"{PITCH_CLASSES[idx-12]}_minor"

# =========================
# Music theory helpers
# =========================

# Scale intervals (in semitones from tonic)
MAJOR_SCALE = [0, 2, 4, 5, 7, 9, 11]
NAT_MINOR_SCALE = [0, 2, 3, 5, 7, 8, 10]

# Common progressions in scale degrees (1..7)
# Use roman degree indices: 1=tonic, 2=supertonic, ...
PROGRESSIONS_MAJOR = [
    [1, 4, 5, 1],      # I-IV-V-I
    [1, 6, 4, 5],      # I-vi-IV-V (pop)
    [2, 5, 1],         # ii-V-I
    [1, 5, 6, 4],      # I-V-vi-IV
    [6, 4, 1, 5],      # vi-IV-I-V
]

PROGRESSIONS_MINOR = [
    [1, 6, 7, 1],      # i-VI-VII-i (natural minor flavor)
    [1, 4, 5, 1],      # i-iv-v-i (strict natural minor v)
    [1, 7, 6, 7],      # i-VII-VI-VII
    [1, 6, 3, 7],      # i-VI-III-VII
]


In [ ]:
def chroma_vec(pitches):
    """Return 12-dim multi-hot chroma vector from pitch classes."""
    v = np.zeros(12, dtype=np.float32)
    for pc in pitches:
        v[pc % 12] = 1.0
    return v

def build_scale(tonic_pc: int, is_major: bool):
    intervals = MAJOR_SCALE if is_major else NAT_MINOR_SCALE
    return [(tonic_pc + i) % 12 for i in intervals]

def diatonic_triad(scale_pcs, degree: int):
    """
    Build a diatonic triad from a 7-note scale.
    degree: 1..7
    triad: scale[deg-1], scale[deg+1], scale[deg+3] (mod 7)
    """
    i = (degree - 1) % 7
    return [scale_pcs[i], scale_pcs[(i+2) % 7], scale_pcs[(i+4) % 7]]

def diatonic_seventh(scale_pcs, degree: int):
    """
    Diatonic 7th chord: add the 7th scale degree tone (stacked thirds)
    """
    tri = diatonic_triad(scale_pcs, degree)
    i = (degree - 1) % 7
    seventh = scale_pcs[(i+6) % 7]
    return tri + [seventh]

def maybe_add_noise(v, p_drop=0.03, p_spur=0.02):
    """
    Optional 'student-like' imperfections:
    - occasionally drop a chord tone
    - occasionally add a wrong extra pitch class
    """
    v = v.copy()
    # drop a 1
    if random.random() < p_drop:
        ones = np.where(v > 0.5)[0]
        if len(ones) > 0:
            v[random.choice(list(ones))] = 0.0
    # add a spurious
    if random.random() < p_spur:
        zeros = np.where(v < 0.5)[0]
        if len(zeros) > 0:
            v[random.choice(list(zeros))] = 1.0
    return v

In [ ]:
def generate_chord_progression_example(key_idx: int) -> np.ndarray:
    """
    Returns an array shape (SEQ_LEN, 12) of chroma frames.
    """
    is_major = key_idx < 12
    tonic_pc = key_idx if is_major else key_idx - 12
    scale = build_scale(tonic_pc, is_major)

    prog = random.choice(PROGRESSIONS_MAJOR if is_major else PROGRESSIONS_MINOR)

    seq = []
    while len(seq) < SEQ_LEN:
        degree = random.choice(prog)
        # Mix triads and 7ths
        chord_pcs = diatonic_seventh(scale, degree) if random.random() < 0.35 else diatonic_triad(scale, degree)
        frame = chroma_vec(chord_pcs)
        frame = maybe_add_noise(frame)

        hold = random.randint(CHUNK_MIN, CHUNK_MAX)
        seq.extend([frame] * hold)

        # Occasionally insert a passing scale tone as a single-note event
        if random.random() < 0.25 and len(seq) < SEQ_LEN:
            note_pc = random.choice(scale)
            note_frame = chroma_vec([note_pc])
            note_frame = maybe_add_noise(note_frame, p_drop=0.01, p_spur=0.01)
            seq.extend([note_frame] * random.randint(1, 2))

    return np.stack(seq[:SEQ_LEN], axis=0)

def generate_scale_run_example(key_idx: int) -> np.ndarray:
    """
    Sequence of mostly single-note scale tones (like a scale exercise),
    with occasional tonic triad hits.
    """
    is_major = key_idx < 12
    tonic_pc = key_idx if is_major else key_idx - 12
    scale = build_scale(tonic_pc, is_major)

    # Decide run direction and step pattern
    ascending = random.random() < 0.5
    order = scale if ascending else list(reversed(scale))

    seq = []
    i = 0
    while len(seq) < SEQ_LEN:
        pc = order[i % len(order)]
        frame = chroma_vec([pc])
        frame = maybe_add_noise(frame, p_drop=0.02, p_spur=0.02)

        hold = random.randint(1, 3)
        seq.extend([frame] * hold)

        # occasional tonic triad "checkpoint"
        if random.random() < 0.12 and len(seq) < SEQ_LEN:
            tri = diatonic_triad(scale, 1)
            tri_frame = maybe_add_noise(chroma_vec(tri))
            seq.extend([tri_frame] * random.randint(2, 5))

        i += 1

    return np.stack(seq[:SEQ_LEN], axis=0)

def generate_example(key_idx: int) -> np.ndarray:
    # Mix types so model learns both harmony + scale exercises
    if random.random() < 0.6:
        return generate_chord_progression_example(key_idx)
    else:
        return generate_scale_run_example(key_idx)

def generate_split(samples_per_key: int):
    X_list, y_list = [], []
    for key_idx in range(24):
        for _ in range(samples_per_key):
            X_list.append(generate_example(key_idx))
            y_list.append(key_idx)
    X = np.stack(X_list, axis=0).astype(np.float32)  # (N, T, 12)
    y = np.array(y_list, dtype=np.int64)             # (N,)
    # Shuffle
    perm = np.random.permutation(len(y))
    return X[perm], y[perm]

In [ ]:
def save_npz(path, X, y):
    np.savez_compressed(path, X=X, y=y)

def load_npz(path):
    d = np.load(path)
    return d["X"], d["y"]

In [ ]:
class KeyChromaDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, T, 12)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
class KeyDetectionLSTM(nn.Module):
    def __init__(self, input_size=12, hidden_size=128, num_layers=2, num_classes=24, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (B, T, 12)
        out, _ = self.lstm(x)
        last = out[:, -1, :]   # (B, H)
        return self.fc(last)

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    crit = nn.CrossEntropyLoss()
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss = crit(logits, y)
        total_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

def train_model(train_npz="train.npz", val_npz="val.npz"):
    Xtr, ytr = load_npz(train_npz)
    Xva, yva = load_npz(val_npz)

    train_ds = KeyChromaDataset(Xtr, ytr)
    val_ds = KeyChromaDataset(Xva, yva)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = KeyDetectionLSTM().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()

    best_val = 0.0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss = crit(logits, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item() * y.size(0)

        tr_loss = total_loss / len(train_ds)
        va_loss, va_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} | val loss {va_loss:.4f} | val acc {va_acc*100:.2f}%")

        if va_acc > best_val:
            best_val = va_acc
            torch.save(model.state_dict(), "key_detector_synth.pth")

    print(f"Best val acc: {best_val*100:.2f}%")
    print("Saved best model to: key_detector_synth.pth")
    return model


In [ ]:
def main():
    # Generate + save datasets
    Xtr, ytr = generate_split(TRAIN_SAMPLES_PER_KEY)
    Xva, yva = generate_split(VAL_SAMPLES_PER_KEY)

    save_npz("train.npz", Xtr, ytr)
    save_npz("val.npz", Xva, yva)

    print("Saved:", "train.npz", "val.npz")
    print("Train shape:", Xtr.shape, "Val shape:", Xva.shape)
    print("Example label 0 means:", key_index_to_name(0), "| label 12 means:", key_index_to_name(12))

    # Train
    train_model("train.npz", "val.npz")


main()

Saved: train.npz val.npz
Train shape: (48000, 64, 12) Val shape: (9600, 64, 12)
Example label 0 means: C_major | label 12 means: C_minor
Epoch 01 | train loss 1.1231 | val loss 0.7875 | val acc 52.83%
Epoch 02 | train loss 0.8081 | val loss 0.6903 | val acc 64.67%
Epoch 03 | train loss 0.6130 | val loss 0.4992 | val acc 77.89%
Epoch 04 | train loss 0.4413 | val loss 0.3795 | val acc 83.93%
Epoch 05 | train loss 0.3564 | val loss 0.2996 | val acc 87.12%
Epoch 06 | train loss 0.3171 | val loss 0.2916 | val acc 88.27%
Epoch 07 | train loss 0.2742 | val loss 0.3095 | val acc 87.76%
Epoch 08 | train loss 0.2538 | val loss 0.2276 | val acc 90.91%
Epoch 09 | train loss 0.2436 | val loss 0.2191 | val acc 90.90%
Epoch 10 | train loss 0.2315 | val loss 0.2410 | val acc 89.69%
Best val acc: 90.91%
Saved best model to: key_detector_synth.pth


In [ ]:
def test_model(test_npz="train.npz"):
    Xte, yte = load_npz(test_npz)
    test_ds = KeyChromaDataset(Xte, yte)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = KeyDetectionLSTM().to(DEVICE)
    model.load_state_dict(torch.load("key_detector_synth.pth", map_location=DEVICE))

    test_loss, test_acc = evaluate(model, test_loader)

    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc*100:.2f}%")

test_model()



Test loss: 0.2317
Test accuracy: 90.68%


In [ ]:
WINDOW_SEC = 1     # predict every 10 seconds
HOP_SEC = 1

import librosa
import numpy as np
import torch

def load_trained_model():
    model = KeyDetectionLSTM().to(DEVICE)
    model.load_state_dict(torch.load("key_detector_synth.pth", map_location=DEVICE))
    model.eval()
    return model

def audio_to_binary_chroma(y, sr):
    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )  # (12, T)

    chroma = chroma.T  # (T, 12)

    # Normalize each frame
    chroma = chroma / (chroma.max(axis=1, keepdims=True) + 1e-6)

    # Binarize to match synthetic training data
    chroma = (chroma > 0.5).astype(np.float32)

    return chroma


def predict_key_intervals(model, filepath, window_sec=10, hop_sec=5):
    y, sr = librosa.load(filepath, sr=22050)

    window_samples = int(window_sec * sr)
    hop_samples = int(hop_sec * sr)

    predictions = []

    for start in range(0, len(y) - window_samples, hop_samples):
        segment = y[start:start + window_samples]

        # --- Convert segment to chroma ---
        chroma = librosa.feature.chroma_stft(
            y=segment,
            sr=sr,
            n_fft=2048,
            hop_length=512
        ).T  # (time, 12)

        # Normalize
        chroma = chroma / (chroma.max(axis=1, keepdims=True) + 1e-6)

        # Binarize to match training
        chroma = (chroma > 0.5).astype(np.float32)

        # Skip if too short
        if len(chroma) < SEQ_LEN:
            continue

        # Trim or pad to SEQ_LEN
        chroma = chroma[:SEQ_LEN]

        X = torch.tensor(chroma, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = model(X)
            pred = logits.argmax(dim=1).item()

        time_sec = start / sr
        predictions.append((time_sec, pred))

    return predictions

In [ ]:
note_timestamp = []
def print_results(results):
    for t, key_idx in results:
        note_timestamp.append(f"{t:6.1f}s: {key_index_to_name(key_idx)}")
        print(f"{t:6.1f}s → {key_index_to_name(key_idx)}")

def main():
    model = load_trained_model()

    results = predict_key_intervals(
        model,
        "p.mp3",
        window_sec=1.5,
        hop_sec=.5
    )

    print_results(results)


main()

   0.0s → G#_major
   0.5s → G#_major
   1.0s → G#_major
   1.5s → G#_major
   2.0s → G#_major
   2.5s → G#_major
   3.0s → G#_minor
   3.5s → G#_minor
   4.0s → G#_minor
   4.5s → G#_major
   5.0s → G#_major
   5.5s → C_minor
   6.0s → G#_major
   6.5s → C_minor
   7.0s → G#_major
   7.5s → F_minor
   8.0s → F_minor
   8.5s → F_minor
   9.0s → G#_major
   9.5s → F_minor
  10.0s → F_minor
  10.5s → F_major
  11.0s → G#_major
  11.5s → C#_major
  12.0s → C#_major
  12.5s → F#_major
  13.0s → C#_major
  13.5s → C#_major
  14.0s → G#_major
  14.5s → D#_major
  15.0s → D#_major
  15.5s → D#_major
  16.0s → D#_major
  16.5s → D#_major
  17.0s → D#_major
  17.5s → G#_major
  18.0s → D#_major
  18.5s → G#_major
  19.0s → G#_major
  19.5s → G#_major
  20.0s → G#_major
  20.5s → G#_major
  21.0s → G#_major
  21.5s → G#_major
  22.0s → G#_major
  22.5s → G#_major
  23.0s → G#_major
  23.5s → G#_major
  24.0s → F_minor
  24.5s → F_minor
  25.0s → F_minor
  25.5s → C#_minor
  26.0s → C#_major
  26

In [ ]:
from google import genai
from google.genai import types
import os
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
audio_file = client.files.upload(file="p.mp3")

In [ ]:
prompt = """
Rearrange the provided audio into a very beginner-friendly solo acoustic guitar version.

You are also given a timestamp-to-chord reference list.

This reference list is a secondary guidance signal:
- It provides useful suggestions for chord timing and progression.
- It should be considered carefully, but not treated as ground truth.
- The audio remains the primary source of truth.

Relative importance:
- Audio: high confidence
- Reference list: medium confidence
- General music knowledge: fallback only

How to use it:
- Use it to guide your expectations of when chord changes occur.
- Use it to confirm or support what you hear in the audio.
- If the audio is unclear, lean on the reference list more.
- If the audio clearly disagrees, prioritize the audio.

Balance rules:
- Do not ignore the reference list.
- Do not copy it mechanically.
- Your output should loosely align with its structure and timing, but be corrected by the audio where needed.

Instructions:
- Keep the song recognizable and preserve the general harmonic movement.
- Simplify for beginner acoustic guitar using easy playable chords whenever possible.
- Output only the chord changes that actually need to be played.
- Do not list repeated timestamps unless the chord changes.
- Do not add filler timestamps.
- Organize the output by song sections such as Intro, Verse, Pre-Chorus, Chorus, Bridge, and Outro whenever identifiable.
- Use timestamps in m:ss format.
- For each entry, write the beginner-friendly chord first, followed by the approximate original/heard chord in parentheses.
- If the original chord is unclear, make the best musical estimate from the audio and reference list.
- Keep the output concise and performance-ready.

Accuracy and musicality constraints:

- Each chord must last at least 2–4 seconds unless clearly required by the audio.
- Do not create rapid or 1-second chord changes.
- Never assign multiple chords to the same timestamp.
- Only include a new timestamp when the chord actually changes.

- Use a small, consistent set of chords (ideally 3–6 total).
- Keep the progression musically coherent and in a consistent key.
- Avoid random or unrelated chords.

- Prefer common beginner chord progressions (e.g., G–D–Em–C).
- Avoid unnecessary complexity or over-detection.

- Align chord changes with musical phrases, not arbitrary timestamps.
- Treat sections independently

Consistency and progression rules:

Before generating the final output, you must:

1. Identify the most likely key of the song.
2. Determine a small set of core chords (3–6 chords) that best represent the song.
3. Infer a repeating chord progression pattern (e.g., I–V–vi–IV).

Then:
- Use this progression as the foundation for all sections.
- Keep chord choices consistent across the entire song.
- Avoid changing chords unless clearly required.

- Do NOT re-evaluate chords independently at each timestamp.
- Base all chord decisions on the established progression.

- If a section repeats (e.g., Verse 1 and Verse 2), reuse the same chord progression unless the audio clearly changes.
- Normalize chord timing to a steady rhythm (roughly every 2–4 seconds or per musical phrase).

Output exactly this structure:

Chords You'll Need
- [list only the beginner-friendly guitar chords used]

Song Structure with Chords and Timestamps

**[Section Name]**
* 0:03: G (Ab Major)
* 0:06: D (Eb Major)

Only output these two sections and nothing else.
"""


response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[note_timestamp, audio_file, prompt],
    config=types.GenerateContentConfig(temperature=0.05, top_p=0.8, top_k=20),
)
#\"Perfect\" by Ed Sheeran, given in
print(response.text)
chords_to_play = response.text

Chords You'll Need
- G
- Em
- C
- D

Song Structure with Chords and Timestamps

**Intro**
* 0:00: G (G# Major)

**Verse 1**
* 0:03: G (G# Major)
* 0:07: Em (F minor)
* 0:11: C (C# Major)
* 0:15: G (G# Major)
* 0:18: Em (F minor)
* 0:22: C (C# Major)
* 0:26: D (D# Major)

**Pre-Chorus**
* 0:32: G (G# Major)
* 0:36: Em (F minor)
* 0:40: C (C# Major)

**Chorus**
* 0:44: G (G# Major)
* 0:48: Em (C minor)
* 0:52: C (C# Major)
* 0:56: D (D# Major)
* 1:00: G (G# Major)

**Verse 2**
* 1:02: G (G# Major)
* 1:08: Em (F minor)
* 1:14: C (C# Major)
* 1:18: D (D# Major)

**Pre-Chorus**
* 1:21: G (G# Major)
* 1:25: Em (F minor)

**Chorus**
* 1:28: C (C# Major)
* 1:31: D (D# Major)
* 1:34: G (G# Major)

**Bridge**
* 1:41: G (G# Major)
* 1:45: Em (F minor)
* 1:49: C (C# Major)
* 1:53: D (D# Major)
* 1:57: G (G# Major)
* 2:01: Em (F minor)
* 2:05: C (C# Major)
* 2:09: D (D# Major)

**Pre-Chorus**
* 2:11: G (G# Major)
* 2:15: Em (F minor)

**Chorus**
* 2:19: C (C# Major)
* 2:23: D (D# Major)
* 2:25: G (

In [ ]:
prompt2 = """
You are given:
1. An audio file of a song
2. A structured chord timeline with timestamps (generated previously)

Your task is to convert this into a clean, beginner-friendly guitar chord sheet aligned with lyrics, similar to standard chord/lyric sheets.

Instructions:
- Use the audio as the primary reference for timing and phrasing.
- Use the provided chord timeline as guidance, not strict truth.
- Align chords above the exact words where the chord changes occur.
- Do not include timestamps in the output.
- Only show chords when they change.
- Keep spacing clean and readable.
- Use simple chord names (G, Em, C, D, etc.).
- Keep the structure organized into sections (Intro, Verse, Pre-Chorus, Chorus, etc.).
- Do not overfill chords — only place them where musically necessary.

Formatting rules:
- Section headers must be in square brackets, e.g. [Verse 1]
- Chords must appear directly above the corresponding lyrics
- Do NOT repeat chords unnecessarily
- Keep everything minimal and clean

Example format:

[Intro]
G

[Verse 1]
        G              Em
I found a love for me
        C                D
Darling, just dive right in and follow my lead

        G              Em
Well, I found a girl beautiful and sweet
        C                   D
I never knew you were the someone waiting for me

[Pre-Chorus]
        G
Cause we were just kids when we...
"""
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[chords_to_play, audio_file, prompt2],
    config=types.GenerateContentConfig(temperature=0.05, top_p=0.8, top_k=20),
)

print(response.text)


[Intro]
G

[Verse 1]
        G              Em
I found a love for me
        C                     G
Darling, just dive right in and follow my lead
        Em
Well, I found a girl
        C                   D
beautiful and sweet, I never knew you were the someone waiting for me

[Pre-Chorus]
        G                               Em
Cause we were just kids when we fell in love
                     C
Not knowing what it was, I will not give you up

[Chorus]
        G
this time
        Em
Darling, just kiss me slow
        C
Your heart is all I own
        D
And in your eyes, you're home and mine

[Verse 2]
G
Baby, I'm dancing in the dark
        Em
With you between my arms
        C
Barefoot on the grass
        D
Listening to our favorite song

[Pre-Chorus]
        G                               Em
When you said you looked a mess, I whispered underneath my breath

[Chorus]
        C
But you heard it, darling, you look
D             G
perfect tonight

[Bridge]
        G
Well, I found